#### **1️⃣ What is a Source in dbt? (Plain English)**

In **dbt**, a **source** represents **raw data tables that already exist in your data warehouse.**

👉 These tables are **NOT created by dbt**

👉 They usually come from **external systems** like:

- Production databases

- SaaS tools (Airbnb, Salesforce, Stripe, etc.)

- Data ingestion tools (Fivetran, Airbyte, Stitch)

**In short:**

> Sources = entry point of data into dbt

-------

**Simple analogy 🧠**

Think of dbt like a **factory**:

- **🏭 Factory** → dbt project

- **📦 Raw materials** → Sources

- **🔧 Machines** → Models

- **📊 Finished products** → Analytics tables

dbt **does NOT create raw materials**, it only **refines them**.


-------------

#### **2️⃣ Why do we need Sources at all?**

You might ask:

❓ “Why can’t I just write:

> `SELECT * FROM raw.raw_listings` ?”

You can, but that’s **bad practice.**


#### **Problems without sources ❌**

| Problem             | Why it’s bad                         |
| ------------------- | ------------------------------------ |
| Hard-coded schemas  | Breaks when schema changes           |
| No documentation    | No idea what the table means         |
| No freshness checks | You don’t know if data is stale      |
| No lineage          | dbt can’t track where data came from |
| No testing          | Missing tables go unnoticed          |

--------

**What sources solve ✅**

| Feature            | Benefit                             |
| ------------------ | ----------------------------------- |
| Central definition | One place to manage raw tables      |
| Auto documentation | dbt Docs shows raw data clearly     |
| Freshness checks   | Detect broken pipelines             |
| Lineage tracking   | See raw → staging → marts           |
| Safer refactoring  | Schema changes won’t silently break |


**👉 Sources bring discipline to raw data usage**

--------

#### **3️⃣ Where are Sources defined in dbt?**

Sources are defined in **YAML files**, usually inside:

In [ ]:
models/
 └── sources/
      └── sources.yml

⚠️ Important:

- **SQL files → transformations**

- **YAML files → metadata (sources, tests, docs)**

---------

#### **4️⃣ Basic Source Definition (Example)**

Let’s say in **Snowflake** you have:

- Database: `AIRBNB`

- Schema: `RAW`

- Tables:

    - `RAW_LISTINGS`

    - `RAW_HOSTS`

    - `RAW_REVIEWS`

-----------

`sources.yml`

In [ ]:
version: 2

sources:
  - name: airbnb
    database: AIRBNB
    schema: RAW

    tables:
      - name: listings
        identifier: RAW_LISTINGS

      - name: hosts
        identifier: RAW_HOSTS

      - name: reviews
        identifier: RAW_REVIEWS

**🔍 Break it down line by line**

`version: 2`

- Required by dbt for schema files

- Enables testing + documentation


`sources:`

- Top-level key

- You can define **multiple source systems**

`name: airbnb`

- Logical name used **inside dbt**

- Used later as:

In [ ]:
{{ source('airbnb', 'listings') }}

👉 This is **NOT** a schema name

👉 It’s a **logical grouping**

`database: AIRBNB`

- Optional but recommended

- Explicitly tells dbt which database to use


`schema: RAW`

- Where raw tables physically live

- If schema changes, update it **once here**


`tables:`

- Each table inside the schema.

`name: listings`

- Logical table name used in dbt

- Used in **source()**


`identifier: RAW_LISTINGS`

- Actual physical table name in Snowflake

- Useful when naming conventions differ


-----------

#### **5️⃣ How do you USE a Source in SQL?**

**❌ Bad way (hard-coding)**

In [ ]:
SELECT * FROM RAW.RAW_LISTINGS

Problems:

- dbt doesn’t know lineage

- No freshness checks

- No docs

**✅ Correct dbt way (using source())**

In [ ]:
SELECT *
FROM {{ source('airbnb', 'listings') }}

**What dbt does internally 🧠**

It compiles this into:

In [ ]:
SELECT *
FROM AIRBNB.RAW.RAW_LISTINGS

But now dbt:

- Tracks lineage

- Validates table existence

- Enables freshness & tests

- Shows it in dbt Docs

--------

#### **6️⃣ Sources vs ref() (Very Important)**

| Aspect         | source()        | ref()                |
| -------------- | --------------- | -------------------- |
| Points to      | Raw tables      | dbt models           |
| Created by dbt | ❌ No            | ✅ Yes                |
| Used for       | Ingestion layer | Transformation layer |
| Dependency     | External        | Internal             |
| Example        | `RAW_HOSTS`     | `stg_hosts`          |


------------

**Correct flow in dbt 🔁**

In [ ]:
SOURCE (raw)
   ↓
STAGING MODEL
   ↓
DIM / FACT MODELS


Example:

In [ ]:
-- stg_hosts.sql
SELECT *
FROM {{ source('airbnb', 'hosts') }}

In [ ]:
-- dim_hosts.sql
SELECT *
FROM {{ ref('stg_hosts') }}

**👉 Rule of thumb:**

- **Raw → source()**

- **Anything built by dbt → ref()**


-----------

#### **7️⃣ Freshness Checks (Why sources are powerful)**

You can ask dbt:

> “Is my raw data fresh or broken?”

**Example**

In [ ]:
tables:
  - name: reviews
    freshness:
      warn_after:
        count: 12
        period: hour
      error_after:
        count: 24
        period: hour
    loaded_at_field: review_date

**Meaning:**

- ⚠️ Warn if data is older than 12 hours

- ❌ Fail if older than 24 hours

Run:

In [ ]:
dbt source freshness

👉 This is **impossible without sources**

---------

**8️⃣ What happens if you DON’T use sources?**

Let’s say:

- RAW schema changes to RAW_V2

- You hard-coded SQL everywhere

💥 Result:

- 20 models break

- No clear error source

- Manual debugging hell



**With sources ✅**

Just update:

In [ ]:
schema: RAW_V2

🎉 Everything works again.

---------

#### **9️⃣ Sources in dbt Docs & Lineage**

When you run:

In [ ]:
dbt docs generate
dbt docs serve

You will see:

- Raw tables documented

- Columns (if added)

- Lineage graph:

In [ ]:
RAW_REVIEWS → stg_reviews → fct_reviews

This is **why sources exist**